# Visual Processing

In [ ]:
from deepface import DeepFace
import cv2
import numpy as np
import multiprocessing
from tqdm import tqdm  # For progress tracking

# Load DeepFace model once to avoid reloading it multiple times
deepface_model = "Facenet"

def extract_deepface_features(video_path):
    cap = cv2.VideoCapture(video_path)
    features = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 300:  # Limit processing to first 300 frames
            break

        if frame_count % 10 == 0:  # Process every 10th frame
            try:
                frame = cv2.resize(frame, (224, 224))  # Resize to speed up processing
                embedding = DeepFace.represent(frame, model_name=deepface_model, enforce_detection=False)
                features.append(np.array(embedding[0]["embedding"]))
            except Exception:
                continue

        frame_count += 1

    cap.release()
    return np.mean(features, axis=0) if features else np.zeros(128)  # 128-dimensional vector

# Parallel processing function
def process_videos(video_paths):
    with multiprocessing.Pool(processes=multiprocessing.cpu_count() - 1) as pool:
        features = list(tqdm(pool.imap(extract_deepface_features, video_paths), total=len(video_paths)))
    return features

# Extract visual features in parallel
video_paths = labels_df["video_path"].tolist()
labels_df["visual_features"] = process_videos(video_paths)

# Convert features into a NumPy array and save
visual_features_matrix = np.vstack(labels_df["visual_features"].values)
np.save("visual_features.npy", visual_features_matrix)

print("✅ Visual feature extraction completed and saved!")


# Audio Processing

In [1]:
import os

def extract_audio(video_path):
    audio_path = video_path.replace(".mp4", ".wav")
    if not os.path.exists(audio_path):  # Avoid reprocessing existing files
        os.system(f'ffmpeg -i "{video_path}" -ac 1 -ar 16000 "{audio_path}" -loglevel error')
    return audio_path if os.path.exists(audio_path) else None  # Return extracted path

# Apply function to extract audio for all videos
labels_df["audio_path"] = labels_df["video_path"].apply(extract_audio)

# Check if all audio files were extracted successfully
print("Missing audio files:", labels_df["audio_path"].isna().sum())


NameError: name 'labels_df' is not defined

In [ ]:
import librosa
import numpy as np

def extract_audio_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=74)  # Extract 13 MFCC coefficients
        return np.mean(mfcc, axis=1)  # Average over time
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return np.zeros(74)  # Return zero vector if audio cannot be processed

# Extract features for each audio file
labels_df["audio_features"] = labels_df["audio_path"].apply(extract_audio_features)

# Save extracted features
audio_features_matrix = np.vstack(labels_df["audio_features"].values)
np.save("audio_features.npy", audio_features_matrix)
